# Topic 2: Executor Configuration & Cluster Sizing

> **Phase 4 → Week 6 → Topic 2**

---

## Driver vs Executor

```
┌─────────────────────────────────────────────────────────────────┐
│                        Spark Application                        │
│                                                                 │
│  ┌─────────────────────────────────┐                           │
│  │         DRIVER                  │                           │
│  │  • Your SparkSession lives here │                           │
│  │  • Builds the DAG               │                           │
│  │  • Schedules tasks              │                           │
│  │  • Collects results (collect()) │                           │
│  │  • Large collect() → OOM driver │                           │
│  │  spark.driver.memory = 2-4g     │                           │
│  └────────────────┬────────────────┘                           │
│                   │ schedules tasks                             │
│    ┌──────────────┼──────────────┐                             │
│    ▼              ▼              ▼                              │
│  ┌──────┐      ┌──────┐      ┌──────┐                         │
│  │Exec 1│      │Exec 2│      │Exec 3│                         │
│  │4 core│      │4 core│      │4 core│  ← runs actual tasks    │
│  │ 8GB  │      │ 8GB  │      │ 8GB  │  ← stores shuffle data  │
│  └──────┘      └──────┘      └──────┘  ← holds cache          │
└─────────────────────────────────────────────────────────────────┘
```

---

## The 3 Knobs That Matter Most

```
spark.executor.instances  = N    (how many executors)
spark.executor.cores      = C    (vCPUs per executor)
spark.executor.memory     = M    (JVM heap per executor)

Total parallelism = N × C   (number of tasks that can run simultaneously)
Total memory      = N × M
```

---

## Sizing Anti-Patterns

```
Anti-pattern 1: TINY executors (many small)
  instances=100, cores=1, memory=1g
  Problem: JVM startup overhead × 100; broadcast tables tiny; too much coordination

Anti-pattern 2: FAT executors (one huge)
  instances=1, cores=32, memory=200g
  Problem: All eggs one basket; GC pauses affect all 32 tasks; HDFS blocks limited
  per core; no fault tolerance (one executor failure = entire job fails)

Sweet spot: MEDIUM executors
  instances=20, cores=4-5, memory=8-16g
  Balances: parallelism, GC, HDFS throughput, fault tolerance
```

---

## Interview Cheat Sheet

**Q: What's the difference between driver and executor memory?**
> The driver (`spark.driver.memory`) runs the SparkSession, builds the DAG, and collects results. It needs enough memory to hold the result of `collect()` calls and broadcast variables. Executors (`spark.executor.memory`) do the actual compute work — sorting, aggregating, shuffling, caching. Increase driver memory if `collect()` OOMs; increase executor memory if tasks OOM during computation.

**Q: How many cores per executor is optimal?**
> 4-5 cores per executor is the industry rule of thumb. More than 5 and HDFS I/O throughput per core degrades (HDFS has connection limits per node). Also, with too many cores, a single GC pause freezes all tasks on that executor simultaneously — 5 cores means max 5 tasks lost to one GC pause instead of 32.

**Q: What is `spark.default.parallelism` vs `spark.sql.shuffle.partitions`?**
> `spark.default.parallelism` controls default partition count for RDD operations (parallelize, textFile). `spark.sql.shuffle.partitions` controls partitions after DataFrame shuffles (groupBy, join). For SQL/DataFrame workloads, only `shuffle.partitions` matters. Set it to 2-4× total executor cores. AQE can auto-tune this at runtime.

**Q: What is dynamic allocation?**
> Dynamic allocation (`spark.dynamicAllocation.enabled=true`) lets Spark scale executor count up and down based on workload. Idle executors are released; new ones are requested when tasks queue up. Useful on shared clusters. Requires an external shuffle service so released executors don't lose their shuffle data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week6 - Executor Config") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | local mode (simulates 4 cores)")

## Part 1: Inspecting the Current Configuration

In [ ]:
sc = spark.sparkContext

print("=== Cluster Info ===")
print(f"  Master:           {sc.master}")
print(f"  App Name:         {sc.appName}")
print(f"  Spark Version:    {sc.version}")
print(f"  Default Parallelism: {sc.defaultParallelism}")
print()

print("=== Key Config ===")
config_keys = [
    "spark.executor.instances",
    "spark.executor.cores",
    "spark.executor.memory",
    "spark.executor.memoryOverhead",
    "spark.driver.memory",
    "spark.driver.cores",
    "spark.sql.shuffle.partitions",
    "spark.default.parallelism",
    "spark.dynamicAllocation.enabled",
]
for k in config_keys:
    try:
        v = spark.conf.get(k)
    except Exception:
        v = "not set"
    print(f"  {k:<45s} = {v}")

## Part 2: The Cluster Sizing Calculator

In [ ]:
def size_cluster(num_nodes: int, vcpus_per_node: int, ram_gb_per_node: int,
                 cores_per_executor: int = 4):
    """
    Industry-standard Spark cluster sizing calculator.
    Leaves 1 core and 1GB per node for OS/YARN/Datanode daemons.
    """
    cores_for_spark  = vcpus_per_node - 1
    ram_for_spark_gb = ram_gb_per_node - 1

    execs_per_node   = max(1, cores_for_spark // cores_per_executor)
    total_executors  = execs_per_node * num_nodes
    total_cores      = total_executors * cores_per_executor

    # Memory per executor = available RAM / executors per node
    # Then subtract 10% overhead
    raw_mem_per_exec_gb = ram_for_spark_gb / execs_per_node
    heap_per_exec_gb    = int(raw_mem_per_exec_gb * 0.9)  # 10% for overhead

    shuffle_partitions  = total_cores * 3  # rule: 2-4x total cores

    print(f"\nCluster: {num_nodes} nodes × {vcpus_per_node} vCPU × {ram_gb_per_node}GB")
    print(f"─────────────────────────────────────────────────")
    print(f"  spark.executor.instances    = {total_executors}")
    print(f"  spark.executor.cores        = {cores_per_executor}")
    print(f"  spark.executor.memory       = {heap_per_exec_gb}g")
    print(f"  spark.sql.shuffle.partitions= {shuffle_partitions}")
    print(f"  ─────────────────────────────────────────────")
    print(f"  Total executor cores:   {total_cores}")
    print(f"  Total executor memory:  {total_executors * heap_per_exec_gb}GB")
    print(f"  Max parallel tasks:     {total_cores}")

# Common cluster configurations
size_cluster(num_nodes=5, vcpus_per_node=16, ram_gb_per_node=64)
size_cluster(num_nodes=10, vcpus_per_node=8, ram_gb_per_node=32)
size_cluster(num_nodes=20, vcpus_per_node=4, ram_gb_per_node=16)

## Part 3: Parallelism — shuffle.partitions vs default.parallelism

In [ ]:
# Effect of shuffle.partitions on performance
import time

large_df = spark.range(1_000_000) \
    .withColumn("key",   (F.col("id") % 1000).cast("string")) \
    .withColumn("value", F.rand() * 100)

def timed_groupby(shuffle_partitions: int):
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions))
    t0 = time.time()
    count = large_df.groupBy("key").agg(F.sum("value")).count()
    elapsed = time.time() - t0
    print(f"  shuffle.partitions={shuffle_partitions:>4d} → {elapsed:.3f}s  ({count} groups)")

print("Effect of spark.sql.shuffle.partitions (4 local cores):")
for n in [1, 2, 4, 8, 20, 50, 200]:
    timed_groupby(n)

print()
print("Observation:")
print("  • Too few: large partitions → slow per-task, not using all cores")
print("  • Too many: tiny partitions → task scheduling overhead dominates")
print("  • Sweet spot: 2-4× total cores (here: 4 cores → 8-16 is good)")
spark.conf.set("spark.sql.shuffle.partitions", "8")

## Part 4: Dynamic Allocation

In [ ]:
print("""
Dynamic Allocation Configuration:
─────────────────────────────────────────────────────────────────
spark.dynamicAllocation.enabled              = true
spark.dynamicAllocation.minExecutors         = 2    (never release all)
spark.dynamicAllocation.maxExecutors         = 50   (cap growth)
spark.dynamicAllocation.initialExecutors     = 5    (start count)
spark.dynamicAllocation.executorIdleTimeout  = 60s  (release after idle)
spark.dynamicAllocation.schedulerBacklogTimeout = 1s (request new after)

REQUIRED for dynamic allocation:
spark.shuffle.service.enabled = true
  → External shuffle service retains shuffle files when executor is released
  → Without it, releasing executors loses their shuffle data → recompute!

OR (Spark 3.2+):
spark.dynamicAllocation.shuffleTracking.enabled = true
  → Tracks which executors hold shuffle data → doesn't release those

When to use:
  ✓ Shared cluster (multiple teams, mixed workloads)
  ✓ Variable workloads (light queries + heavy ETL from same app)
  ✗ Fixed cost clusters (no benefit — pay regardless)
  ✗ ML training (stable parallelism needed; allocation overhead hurts)
─────────────────────────────────────────────────────────────────
""")

## Part 5: spark-submit Configuration Reference

In [ ]:
print("""
spark-submit Template (Production ETL Job)
────────────────────────────────────────────────────────────────
spark-submit \\
  --master yarn \\
  --deploy-mode cluster \\

  # Executor sizing
  --num-executors 20 \\
  --executor-cores 4 \\
  --executor-memory 8g \\
  --driver-memory 4g \\
  --driver-cores 2 \\

  # Memory tuning
  --conf spark.executor.memoryOverhead=1g \\
  --conf spark.memory.fraction=0.6 \\
  --conf spark.memory.storageFraction=0.3 \\

  # Shuffle
  --conf spark.sql.shuffle.partitions=240 \\
  --conf spark.sql.adaptive.enabled=true \\

  # Serialization
  --conf spark.serializer=org.apache.spark.serializer.KryoSerializer \\
  --conf spark.kryoserializer.buffer.max=512m \\

  # Fault tolerance
  --conf spark.task.maxFailures=4 \\
  --conf spark.stage.maxConsecutiveAttempts=4 \\

  # Network
  --conf spark.network.timeout=800s \\
  --conf spark.executor.heartbeatInterval=60s \\

  my_etl_job.py
────────────────────────────────────────────────────────────────
""")

In [ ]:
# SparkSession config in Python (same as spark-submit --conf)
production_spark = SparkSession.builder \
    .appName("ProductionJob") \
    .master("local[4]") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memoryOverhead", "512m") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "48") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()  # returns existing session in same JVM

print("Production-like configuration applied.")
print(f"shuffle.partitions = {production_spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"AQE enabled        = {production_spark.conf.get('spark.sql.adaptive.enabled')}")

## Exercises

1. Size a cluster: 8 nodes × 16 vCPU × 64GB RAM. Calculate `executor.instances`, `executor.cores`, `executor.memory`, and `shuffle.partitions`.
2. What's wrong with this config? `--num-executors 4 --executor-cores 32 --executor-memory 200g` on a 10-node × 4-core × 16GB cluster.
3. A job uses `collect()` and fails with `OutOfMemoryError` on the driver. Which config do you change? Why not the executor memory?
4. When should you enable dynamic allocation? What additional service must be configured?
5. You have 200GB of data to process. `shuffle.partitions=200`. Each task processes ~1GB after shuffle. Is this OK? What would you change?

In [ ]:
# Exercise 1: Size the 8-node cluster
size_cluster(num_nodes=8, vcpus_per_node=16, ram_gb_per_node=64)

In [ ]:
# Exercise 2: What's wrong?
print("""
Problem with: 4 executors × 32 cores × 200GB on a 10-node × 4-core × 16GB cluster:

1. You're requesting 32 cores per executor but each node only has 4 vCPUs!
   A single executor needs 32 cores — impossible on one node.

2. You're requesting 200GB per executor but each node only has 16GB RAM.
   Even if spread across nodes: 4 executors × 200GB = 800GB total requested,
   but cluster only has 10 × 16GB = 160GB. 5x oversubscription.

3. Only 4 executors for a 10-node cluster = most nodes idle.

Correct config for 10 nodes × 4 vCPU × 16GB:
""")
size_cluster(num_nodes=10, vcpus_per_node=4, ram_gb_per_node=16, cores_per_executor=3)